# Atelier 3B — Améliorer un RAG : BM25, recherche hybride et reranking

**Les carnets du RAG · Master 2 — Industry 4.0 / Systèmes informatiques pour le génie de la logistique industrielle et des services**

**Durée indicative :** 90–120 min (approfondissement du Module 3)  
**Prérequis :** TP 1, TP 2 et notions de Recall@k / MRR (TP 3A)  
**Environnement :** Google Colab (CPU suffisant, GPU optionnel), sans clé API et sans Docker.

> **Problématique.** Notre assistant documentaire a été construit. Pourtant, une question portant sur le code `VB-04` ou la règle `FEFO` peut manquer sa source. Comment améliorer la **récupération** sans confondre cette amélioration avec la qualité de la réponse du LLM ?

### À la fin du TP, vous saurez
1. Distinguer **recherche lexicale (BM25)** et **recherche dense** par embeddings.
2. Combiner leurs classements avec la **Reciprocal Rank Fusion (RRF)**.
3. Réordonner les candidats par un **cross-encoder multilingue**.
4. Mesurer **Recall@3, Precision@3, Hit@3, MRR@3 et latence indicative**, par master.
5. Identifier les améliorations réelles, les régressions et les limites du petit corpus.

**Important :** tous les documents et toutes les situations sont **fictifs** ; ils ne constituent pas des procédures de sécurité industrielle ou logistique réelles.

## Carte du laboratoire

```text
                             QUESTION
                               │
                 ┌─────────────┴─────────────┐
                 ▼                           ▼
       BM25 (termes exacts)        Encodeur (sens du texte)
                 │                           │
            Rang lexical                Rang dense
                 └─────────────┬─────────────┘
                               ▼
                        Fusion RRF
                               │
                       Candidats top-n
                               ▼
                Cross-encoder (reranking)
                               │
                       Passages top-k
                               ▼
             Contexte / prompt du RAG → LLM

  Évaluation :  références documentaires annotées → Recall@3 / MRR@3
```

**Périmètre :** nous améliorons ici le **retrieval**. La génération est étudiée au TP 2 et la fidélité des réponses au TP 3A. Un bon score de retrieval n'assure pas automatiquement une réponse exacte.

## 0. Préparer Colab (5 min)

1. Dans Google Colab, ouvrez **Exécution → Modifier le type d’exécution**. Le **CPU** suffit ; un GPU peut accélérer le cross-encoder.
2. Exécutez la cellule d’installation une fois, puis les cellules **de haut en bas**.
3. Le téléchargement initial des modèles nécessite Internet. Les documents fictifs sont **intégrés au notebook** et ne nécessitent aucun accès au dépôt GitHub.

**Installation :** `sentence-transformers` (encodeur et cross-encoder), `rank-bm25`, NumPy, pandas et Matplotlib. Aucun service Qdrant ou Ollama n'est nécessaire pour cet atelier.

In [ ]:
%pip -q install "sentence-transformers>=3,<6" "rank-bm25==0.2.2" numpy pandas matplotlib

In [ ]:
import re, json, time, math, platform
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

K_EVALUATION = 3
N_CANDIDATS = 8      # Chaque moteur propose jusqu'à 8 documents avant fusion
K_RRF = 60          # Paramètre constant de la fusion de rangs
ACTIVER_CROSS_ENCODER = True  # False si temps/connexion insuffisants
SEED = 42
np.random.seed(SEED)
print('Python', platform.python_version(), '· Mode CPU/GPU détecté par PyTorch lors du chargement')

## 1. Reprendre notre corpus commun — deux masters (10 min)

Nous utilisons **exactement les 12 fiches du TP 1 / TP 2**, réparties en deux domaines. Chaque fiche contient un identifiant documentaire, un titre, un texte, un thème et des métadonnées. Nous évaluons au **niveau document** : les résultats citent `IND-01` ou `LOG-02`, et non un identifiant technique arbitraire.

Une fiche = un passage pour cette première comparaison. Pour étudier la granularité du chunking, reprenez le TP 3A.

**Faites apparaître les deux parcours**, puis choisissez le vôtre à l’étape 7.

In [ ]:
CORPUS = json.loads(r'''{"documents":[{"id":"IND-01","titre":"Ligne B — Alarme VB-04","domaine":"Industrie 4.0","theme":"maintenance","texte":"La fiche de maintenance fictive PR-17 concerne une alarme de vibration VB-04 sur le convoyeur de la ligne B. La procédure documentaire demande de consigner l’heure de l’alarme, l’identifiant de la machine et la valeur mesurée. Le responsable habilité vérifie l’historique des alarmes et l’état du dernier entretien. Le document précise que toute intervention physique dépend des consignes de sécurité du site et de l’autorisation du responsable de maintenance. L’assistant documentaire ne doit pas remplacer cette validation.","reference":"IND-01","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"IND-02","titre":"Moteur M-12 — Analyse vibratoire","domaine":"Industrie 4.0","theme":"capteurs","texte":"Le rapport fictif VIB-M12 étudie les vibrations du moteur M-12. Les séries temporelles sont collectées toutes les dix minutes et agrégées par heure pour les rapports. Le rapport distingue vibration moyenne, maximum et dispersion. Une augmentation persistante déclenche une inspection documentaire de l’historique, et non un diagnostic automatique de panne. Les mesures doivent être horodatées et associées à la version du capteur.","reference":"IND-02","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"IND-03","titre":"Capteur de température T-8 — Métrologie","domaine":"Industrie 4.0","theme":"qualité","texte":"La procédure fictive MET-T8 traite la calibration du capteur de température T-8. Chaque enregistrement de calibration comporte une date, un technicien habilité, une valeur de référence et un écart constaté. Le registre conserve les versions successives au lieu d’écraser l’historique. Pour une mesure actuelle, l’agent logiciel doit interroger la base de mesures autorisée ; un corpus documentaire statique ne suffit pas.","reference":"IND-03","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"IND-04","titre":"TRS / OEE — Production","domaine":"Industrie 4.0","theme":"performance","texte":"Le guide fictif PROD-04 définit le taux de rendement synthétique TRS, appelé OEE en anglais, comme le produit disponibilité × performance × qualité. Les trois facteurs sont des proportions comprises entre zéro et un. Pour une disponibilité de 0,90, une performance de 0,95 et une qualité de 0,98, le TRS vaut environ 0,838. Le guide recommande de toujours mentionner la période analysée et la source des données.","reference":"IND-04","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"IND-05","titre":"Maintenance prédictive — Guide interne","domaine":"Industrie 4.0","theme":"maintenance","texte":"Le guide fictif PREV-05 décrit une approche de maintenance prédictive utilisant vibration, température et historique des incidents. Le modèle statistique émet des scores de risque, qui ne doivent pas être présentés comme des certitudes. La validation doit vérifier les dérives de capteurs et les changements de conditions de fonctionnement. Les documents décrivent les modèles, tandis que les mesures récentes sont consultées dans un système de supervision.","reference":"IND-05","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"IND-06","titre":"Traçabilité des lots — Atelier","domaine":"Industrie 4.0","theme":"traçabilité","texte":"Le document fictif TRACE-06 explique que chaque lot de production possède un identifiant unique. Les évènements de fabrication sont associés au lot, à une machine, à un créneau horaire et à une version de procédure. Lorsqu’une anomalie qualité est signalée, l’équipe cherche les lots concernés et la documentation applicable à la date concernée. Une réponse RAG doit citer la version du document plutôt que mélanger plusieurs révisions.","reference":"IND-06","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"LOG-01","titre":"Réception des marchandises — Entrepôt","domaine":"Logistique et services","theme":"entrepôt","texte":"La procédure fictive REC-01 indique que toute réception doit être associée à un bon de livraison et à une commande attendue. En cas d’écart de quantité, l’agent enregistre une anomalie dans le WMS et la transmet au responsable de quai. Le stock n’est mis à jour qu’après la validation prévue par le processus interne. Les documents d’exploitation précisent les rôles ; l’état réel des commandes provient du WMS.","reference":"LOG-01","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"LOG-02","titre":"Règle FEFO — Gestion des stocks","domaine":"Logistique et services","theme":"stocks","texte":"Le référentiel fictif STOCK-02 présente la méthode FEFO : First Expired, First Out. Pour les produits soumis à date limite, les unités dont la date d’expiration est la plus proche sont préparées en priorité, sous réserve des contraintes de qualité et de disponibilité. Le texte distingue FEFO et FIFO, qui suit l’ordre d’entrée en stock. L’assistant doit citer la politique de stock effectivement applicable à la catégorie de produits.","reference":"LOG-02","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"LOG-03","titre":"Quais de chargement — Ordonnancement","domaine":"Logistique et services","theme":"transport","texte":"Le manuel fictif QUAI-03 décrit l’ordonnancement des quais de chargement. Chaque créneau est lié à un transporteur, à une unité de chargement et à un niveau de priorité. Lorsqu’un camion arrive en retard, la planification est ajustée par un agent habilité dans le TMS. Un RAG documentaire peut expliquer les règles du planning mais ne doit pas inventer l’horaire actuel d’un camion.","reference":"LOG-03","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"LOG-04","titre":"Inventaire tournant — WMS","domaine":"Logistique et services","theme":"stocks","texte":"La procédure fictive INV-04 présente l’inventaire tournant : une fraction des emplacements est contrôlée régulièrement sans immobiliser tout l’entrepôt. Les écarts entre stock physique et stock théorique sont enregistrés dans le WMS puis soumis à validation. Les comptages sont rattachés à une date, une zone et un opérateur autorisé. Les réponses fondées sur ce document décrivent la procédure mais pas les quantités réelles.","reference":"LOG-04","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"LOG-05","titre":"Livraison — Gestion des incidents","domaine":"Logistique et services","theme":"services","texte":"La politique fictive LIV-05 indique qu’un incident de livraison reçoit un numéro de ticket et une catégorie : retard, colis endommagé ou adresse incorrecte. Le service client consulte le TMS, vérifie la preuve de livraison lorsqu’elle existe et transmet l’incident au responsable concerné. Un modèle de langage peut expliquer les étapes du dossier, mais seul le système opérationnel donne le statut courant.","reference":"LOG-05","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"},{"id":"LOG-06","titre":"Retours clients — Logistique inverse","domaine":"Logistique et services","theme":"services","texte":"La procédure fictive RET-06 décrit les retours clients. Le retour est identifié par une autorisation, une référence de commande et un motif. La réception de l’article est distinguée de la décision de remise en stock. Une inspection qualité peut être nécessaire selon l’état du produit. La base documentaire décrit les conditions de traitement ; le statut du retour est maintenu dans le système transactionnel.","reference":"LOG-06","version":"V1 · corpus pédagogique fictif","type":"Fiche interne fictive"}],"questions":[{"question":"Que prévoit le document PR-17 pour l’alarme VB-04 sur le convoyeur ?","sources_attendues":["IND-01"]},{"question":"À quelle fréquence les mesures vibratoires du moteur M-12 sont-elles collectées ?","sources_attendues":["IND-02"]},{"question":"Quels éléments sont consignés lors de la calibration du capteur T-8 ?","sources_attendues":["IND-03"]},{"question":"Comment calcule-t-on le TRS ou OEE ?","sources_attendues":["IND-04"]},{"question":"Quelles sources de données utilise la maintenance prédictive ?","sources_attendues":["IND-05"]},{"question":"Comment la traçabilité des lots prend-elle en compte les versions des procédures ?","sources_attendues":["IND-06"]},{"question":"Que faire lorsqu’une quantité reçue est différente de la commande ?","sources_attendues":["LOG-01"]},{"question":"Quelle différence existe entre FEFO et FIFO ?","sources_attendues":["LOG-02"]},{"question":"Comment est géré un camion en retard sur son créneau de quai ?","sources_attendues":["LOG-03"]},{"question":"En quoi consiste un inventaire tournant ?","sources_attendues":["LOG-04"]},{"question":"Comment catégoriser un incident de livraison dans le processus fictif ?","sources_attendues":["LOG-05"]},{"question":"Quelle différence entre réception d’un retour et remise en stock ?","sources_attendues":["LOG-06"]}]}''')
documents = CORPUS['documents']
docs_par_id = {d['id']: d for d in documents}
ids = [d['id'] for d in documents]
assert len(documents) == 12 and len(set(ids)) == 12

def texte_indexable(d):
    # Indexer aussi le titre : les codes et les sujets peuvent y figurer.
    return f"{d['titre']}. {d['texte']}"

textes = [texte_indexable(d) for d in documents]
df_docs = pd.DataFrame(documents)[['id','titre','domaine','theme']]
display(df_docs)
print('Documents par master :',df_docs.groupby('domaine').size().to_dict())

### Questions annotées : références connues avant l'expérience

Les 12 questions de la formation sont conservées. Nous ajoutons **8 questions de difficulté accrue** (codes exacts et reformulations métier), avec une source de référence explicite. Les questions sans réponse seront traitées séparément, pour ne pas fausser les métriques.

> Attention : le jeu est petit et construit pour l’enseignement. Il permet de **comparer des méthodes sur ce corpus**, pas d'établir qu'un système est supérieur en production.

In [ ]:
QUESTIONS = []
for i, item in enumerate(CORPUS['questions'], start=1):
    ref = item['sources_attendues'][0]
    QUESTIONS.append(dict(qid=f'Q{i:02d}', question=item['question'],
        gold=item['sources_attendues'], domaine=docs_par_id[ref]['domaine'],
        type='référence'))

# Huit nouvelles questions annotées à partir des mêmes fiches.
QUESTIONS_SUP = [
    ('IND-01', 'Sur quel convoyeur est mentionnée l’alarme VB-04 et que consigner ?', 'code exact'),
    ('IND-03', 'Quel registre doit conserver les anciennes calibrations plutôt que les écraser ?', 'reformulation'),
    ('IND-04', 'Par quel calcul associer disponibilité, performance et qualité de production ?', 'reformulation'),
    ('IND-06', 'Comment retrouver les événements associés à un lot et à la version de procédure ?', 'reformulation'),
    ('LOG-02', 'Quel référentiel décrit le principe First Expired, First Out (FEFO) ?', 'code exact'),
    ('LOG-03', 'Quelle procédure ajuste les plages de chargement lorsqu’un camion arrive en retard ?', 'reformulation'),
    ('LOG-04', 'Comment vérifier régulièrement les emplacements sans fermer tout l’entrepôt ?', 'reformulation'),
    ('LOG-05', 'À quel système et quelle preuve se référer lors d’un incident de livraison ?', 'reformulation'),
]
for i, (ref, question, categorie) in enumerate(QUESTIONS_SUP, start=len(QUESTIONS)+1):
    QUESTIONS.append(dict(qid=f'Q{i:02d}', question=question, gold=[ref],
        domaine=docs_par_id[ref]['domaine'], type=categorie))

questions_df = pd.DataFrame(QUESTIONS)
assert len(QUESTIONS)==20 and questions_df.qid.is_unique
assert all(set(q['gold']).issubset(set(ids)) for q in QUESTIONS)
display(questions_df[['qid','domaine','type','question','gold']])
print('Questions par master :',questions_df.groupby('domaine').size().to_dict())

## 2. Baseline : récupérer par le sens (15 min)

Un **bi-encoder** transforme chaque fiche et la question en deux vecteurs d’un même espace. Les documents sont encodés **une fois** ; la question est encodée au moment de la recherche.

On utilise le modèle **multilingue** déjà employé dans les TP précédents :

\[
\mathrm{sim}(q,d)=\frac{e(q)^\top e(d)}{\|e(q)\|_2\,\|e(d)\|_2}.
\]

Comme les vecteurs sont normalisés, le produit scalaire correspond à la similarité cosinus. Sur 12 fiches, une multiplication NumPy est suffisante : le rôle est celui d'une recherche exhaustive, équivalente à l'idée de l'index `FAISS IndexFlatIP` vu précédemment.

**Hypothèse :** les reformulations métiers sont souvent mieux traitées par un encodeur que par une simple correspondance de mots.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELE_EMBEDDING = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
encodeur = SentenceTransformer(MODELE_EMBEDDING)
matrice = np.asarray(encodeur.encode(textes, normalize_embeddings=True,
                                    show_progress_bar=False), dtype=np.float32)
assert matrice.shape[0] == 12
assert np.allclose(np.linalg.norm(matrice,axis=1), 1, atol=1e-3)

def classement_dense(question):
    vecteur_q = np.asarray(encodeur.encode([question],normalize_embeddings=True,
                     show_progress_bar=False),dtype=np.float32)[0]
    scores = matrice @ vecteur_q
    ordre = sorted(range(len(documents)),key=lambda i:(-float(scores[i]),ids[i]))
    return [ids[i] for i in ordre]

EXEMPLE = 'Que dit la procédure concernant le code VB-04 ?'
print('Classement dense pour :', EXEMPLE)
print(classement_dense(EXEMPLE)[:5])

## 3. Recherche lexicale BM25 (10 min)

BM25 valorise la présence de mots de la question, en tenant compte de leur rareté dans le corpus et de la longueur des documents. Il est particulièrement utile pour retrouver **`VB-04`**, **`M-12`**, **`T-8`**, **`WMS`** ou **`FEFO`**.

La tokenisation doit préserver les identifiants composés. `VB-04` doit rester un token unique plutôt que devenir `VB` et `04`.

**Limite :** BM25 ignore les synonymes et les paraphrases qui ne partagent pas de termes. Il ne remplace donc pas les embeddings.

In [ ]:
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r"[\w]+(?:[-_][\w]+)*", flags=re.UNICODE)
def tokeniser(texte):
    return TOKEN_RE.findall(texte.casefold())

assert 'vb-04' in tokeniser('Défaut VB-04 constaté')
assert 'fefo' in tokeniser('Règle FEFO')
bm25 = BM25Okapi([tokeniser(t) for t in textes])

def classement_bm25(question):
    scores = bm25.get_scores(tokeniser(question))
    ordre = sorted(range(len(documents)),key=lambda i:(-float(scores[i]),ids[i]))
    return [ids[i] for i in ordre]

print('BM25 pour :', EXEMPLE)
print(classement_bm25(EXEMPLE)[:5])
print('BM25 avec reformulation :',classement_bm25('Comment prioriser les produits qui expirent bientôt ?')[:5])

## 4. Construire une recherche hybride avec RRF (15 min)

Dense et BM25 produisent des scores sur des **échelles différentes** ; il serait incorrect de les additionner directement. RRF fusionne les **positions dans les classements**, pas les scores :

\[
\mathrm{RRF}(d)=\sum_{j\in\{\mathrm{dense},\mathrm{BM25}\}}\frac{1}{c+\mathrm{rang}_j(d)}.
\]

Par convention, le premier rang vaut **1** (et non 0). Ici `c = 60`. Les documents absents d'une liste de candidats ne reçoivent aucune contribution de cette liste.

**Manipulation :** comparer `dense`, `BM25` et `hybride` sur une question contenant un code précis, puis une question reformulée.

In [ ]:
def fusion_rrf(classements, c=60):
    assert c > 0
    scores = defaultdict(float)
    for classement in classements:
        for rang, identifiant in enumerate(classement,start=1):
            scores[identifiant] += 1.0/(c+rang)
    return sorted(scores,key=lambda d:(-scores[d],d))

# Vérification du calcul : un élément premier dans les deux listes
jeu = fusion_rrf([['IND-01','IND-02'],['IND-01','IND-03']],c=60)
assert jeu[0]=='IND-01'
assert abs(2/(60+1) - (1/(60+1)+1/(60+1))) < 1e-12

def classement_hybride(question, n_candidats=N_CANDIDATS, c=K_RRF):
    rang_dense = classement_dense(question)[:n_candidats]
    rang_bm25 = classement_bm25(question)[:n_candidats]
    return fusion_rrf([rang_dense,rang_bm25],c=c)

for question in [
    'Que signifie le code de vibration VB-04 de la ligne B ?',
    'Comment sortir en priorité les articles dont la date limite approche ?']:
    print('\nQuestion :',question)
    print('Dense   :',classement_dense(question)[:3])
    print('BM25    :',classement_bm25(question)[:3])
    print('Hybride :',classement_hybride(question)[:3])

## 5. Reranking neuronal : faire une seconde lecture des candidats (20 min)

Le **cross-encoder** lit la paire *(question, document)* dans un même modèle et produit un score de pertinence. Il est plus coûteux que la simple recherche dense, car il doit traiter chaque paire. On ne l'applique donc qu'aux **candidats sélectionnés par l'hybride**.

Nous choisissons un **cross-encoder multilingue**, entraîné à classer des passages, adapté à nos questions françaises :

`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`

Les scores du modèle servent **à classer**, pas à estimer directement une probabilité de véracité d'une réponse.

**Limite structurelle :** si une source n'est pas dans les candidats fournis au cross-encoder, ce dernier ne pourra jamais la retrouver. Mesurez donc le rappel des candidats avant de conclure.

**En cas de ressources limitées :** mettre `ACTIVER_CROSS_ENCODER = False` puis relancer cette cellule ; les résultats du reranking seront marqués *non exécuté*, sans être remplacés par de fausses mesures.

In [ ]:
reranker = None
if ACTIVER_CROSS_ENCODER:
    from sentence_transformers import CrossEncoder
    MODELE_RERANK = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
    try:
        reranker = CrossEncoder(MODELE_RERANK)
        print('Cross-encoder chargé :',MODELE_RERANK)
    except Exception as exc:
        print('Reranking neuronal indisponible :',repr(exc))
        print('Vous pouvez poursuivre l’atelier sans inventer de scores de reranking.')

def classement_rerank(question, n_candidats=N_CANDIDATS):
    if reranker is None:
        raise RuntimeError('Cross-encoder non chargé : ne pas calculer de faux résultats.')
    candidats = classement_hybride(question,n_candidats=n_candidats)
    paires = [(question,texte_indexable(docs_par_id[c])) for c in candidats]
    scores = np.asarray(reranker.predict(paires,batch_size=8),dtype=float).reshape(-1)
    ordre = sorted(range(len(candidats)),key=lambda i:(-float(scores[i]),candidats[i]))
    return [candidats[i] for i in ordre]

if reranker is not None:
    print('Avant rerank :',classement_hybride(EXEMPLE)[:5])
    print('Après rerank :',classement_rerank(EXEMPLE)[:5])

## 6. Évaluer rigoureusement le retrieval (15 min)

Les **références attendues** du corpus sont des documents identifiés par `IND-xx` / `LOG-xx`. On évalue à `k=3` :

- **Precision@3** = nombre de documents pertinents retrouvés / 3.
- **Recall@3** = nombre de documents pertinents retrouvés / nombre de références pertinentes.
- **Hit@3** = 1 si une référence est dans le top-3, sinon 0.
- **MRR@3** = inverse du rang de la première réponse pertinente (0 si absente).

Ici, la plupart des questions n'ont **qu'une seule source attendue**. Par conséquent, Hit@3 et Recall@3 sont souvent identiques ; Precision@3 vaut au plus 1/3. Il faut **l'expliquer dans le compte rendu** plutôt que d'interpréter à tort ces chiffres comme une faiblesse du modèle.

La latence mesurée inclut le calcul de la requête et la recherche ; elle dépend de la machine et de l'état de cache. Les temps sont **indicatifs**, pas des benchmarks industriels.

In [ ]:
def metriques_retrieval(classement, gold, k=3):
    assert k>=1
    # Les IDs sont uniques. Une même fiche ne doit jamais compter deux fois.
    rangs = list(dict.fromkeys(classement[:k]))
    attendus = set(gold)
    assert attendus, 'Les questions sans réponse sont traitées à part.'
    touches = len(attendus.intersection(rangs))
    premier_rang = next((i for i,d in enumerate(rangs,start=1) if d in attendus),None)
    return {'precision':touches/k,'recall':touches/len(attendus),
            'hit':float(touches>0), 'mrr':1.0/premier_rang if premier_rang else 0.0}

# Tests unitaires des métriques :
t=metriques_retrieval(['LOG-05','IND-01','LOG-03'],['IND-01'],k=3)
assert t['hit']==1 and t['recall']==1 and abs(t['mrr']-.5)<1e-12
assert abs(t['precision']-1/3)<1e-12
assert metriques_retrieval(['LOG-03','LOG-02','LOG-01'],['IND-01'])['mrr']==0
print('Tests de métriques : OK')

In [ ]:
# Évaluation à conditions comparables (mêmes questions, corpus et k)
configurations = {
    'A — Dense': classement_dense,
    'B — BM25': classement_bm25,
    'C — Hybride RRF': classement_hybride,
}
if reranker is not None:
    configurations['D — Hybride + cross-encoder'] = classement_rerank

resultats = []
for q in QUESTIONS:
    for config, rechercher in configurations.items():
        debut=time.perf_counter()
        rang = rechercher(q['question'])
        duree_ms=(time.perf_counter()-debut)*1000
        m = metriques_retrieval(rang,q['gold'],k=K_EVALUATION)
        resultats.append(dict(qid=q['qid'],master=q['domaine'],type=q['type'],
            configuration=config,question=q['question'],gold=','.join(q['gold']),
            top1=rang[0],top3=', '.join(rang[:K_EVALUATION]),
            latency_ms=duree_ms,**m))

df_eval=pd.DataFrame(resultats)
assert len(df_eval) == len(QUESTIONS)*len(configurations)
resume=df_eval.groupby('configuration',sort=False).agg(
    nb_questions=('qid','nunique'),precision_at_3=('precision','mean'),
    recall_at_3=('recall','mean'),hit_at_3=('hit','mean'),mrr_at_3=('mrr','mean'),
    latence_mediane_ms=('latency_ms','median')).round(3)
display(resume)
print('Valeurs observées pendant VOTRE exécution (aucun résultat prérempli).')

### Comparer les deux masters, puis les types de questions

Le but n’est pas de « faire gagner le cross-encoder », mais de savoir **où il aide et où il dégrade le classement**. Comparez les résultats séparément pour les questions industrielles, logistiques, celles contenant un code et celles reformulées.

In [ ]:
par_master = df_eval.groupby(['master','configuration'],sort=False).agg(
    recall_at_3=('recall','mean'),mrr_at_3=('mrr','mean'),
    latence_mediane_ms=('latency_ms','median')).round(3)
display(par_master)

par_type = df_eval.groupby(['type','configuration'],sort=False).agg(
    nb=('qid','size'),recall_at_3=('recall','mean'),mrr_at_3=('mrr','mean')).round(3)
display(par_type)

fig, axes = plt.subplots(1,2,figsize=(13,4))
resume['recall_at_3'].plot.bar(ax=axes[0],title='Recall@3 par méthode',ylim=(0,1.05))
resume['mrr_at_3'].plot.bar(ax=axes[1],title='MRR@3 par méthode',ylim=(0,1.05))
for a in axes:
    a.set_xlabel(''); a.tick_params(axis='x',labelrotation=20)
plt.tight_layout(); plt.show()

## 7. Diagnostic métier interactif : sélectionnez votre master (10 min)

**Industry 4.0** : alarmes et procédures (`VB-04`, `PR-17`), analyse vibratoire (`M-12`), métrologie (`T-8`), TRS/OEE.

**Logistique industrielle et services** : réception, `WMS`, `FEFO`, quais et créneaux, inventaire tournant, tickets de livraison.

Changez `PARCOURS` et `QID`, puis inspectez les trois premières sources. Vérifiez si les extraits **contiennent réellement la réponse** : une source retrouvée n'est pas automatiquement une preuve suffisante.

In [ ]:
PARCOURS = 'industrie'  # Choix : 'industrie' ou 'logistique'
CHOIX = {'industrie': 'Industrie 4.0', 'logistique':'Logistique et services'}
assert PARCOURS in CHOIX
questions_du_master = [q for q in QUESTIONS if q['domaine']==CHOIX[PARCOURS]]
print(f'Master : {CHOIX[PARCOURS]} · {len(questions_du_master)} questions')
for q in questions_du_master: print(q['qid'],q['question'])

# Choisir un ID affiché ci-dessus ou laisser None pour la première question.
QID = None
q = next((x for x in questions_du_master if x['qid']==QID),questions_du_master[0])
print('\nQuestion choisie :',q['question'],'\nRéférence attendue :',q['gold'])
for label,rechercher in configurations.items():
    print('\n',label)
    for rang,doc_id in enumerate(rechercher(q['question'])[:3],start=1):
        d=docs_par_id[doc_id]
        print(f'  {rang}. [{doc_id}] {d["titre"]} | {d["texte"][:135]}…')

## 8. Pourquoi le reranking peut échouer (5 min)

Le reranker **ne peut pas faire revenir une source exclue de son vivier de candidats**. Vérifions le rappel de la liste hybride avant le reranking et réfléchissons au paramètre `N_CANDIDATS` : un vivier plus grand augmente le rappel potentiel, mais accroît le coût du cross-encoder.

**Question de réflexion :** augmenter `N_CANDIDATS` de 4 à 8 puis à 12 améliore-t-il toujours Recall@3 ? Justifiez à partir de vos expériences.

In [ ]:
for n in [4,8,12]:
    taux=[]
    for q in QUESTIONS:
        pool=classement_hybride(q['question'],n_candidats=n)
        taux.append(float(bool(set(q['gold']) & set(pool))))
    print(f'Vivier (par méthode) n={n:2d} : Hit du pool = {np.mean(taux):.3f}')
print('Attention : les rangs finaux et le score du reranker peuvent évoluer indépendamment du rappel du vivier.')

## 9. Réinjecter les documents retrouvés dans un RAG (5 min)

Le **retrieval amélioré** n'est qu'une étape : son top-k doit être transmis au générateur avec les **titres, références et textes**. Le modèle de génération ne reçoit pas les vecteurs d'embeddings.

Cette cellule prépare un **prompt documenté**, sans appeler de LLM. Pour la génération effective, on peut réutiliser le LLM du TP 2 ou de l'Atelier 2B. Une citation valide doit être vérifiée dans le document source.

In [ ]:
def prompt_documente(question, ids_sources):
    blocs=[]
    for ident in ids_sources:
        d=docs_par_id[ident]
        blocs.append(f"SOURCE [{ident}] — {d['titre']}\n{d['texte']}")
    contexte='\n\n'.join(blocs)
    return ("Tu es un assistant documentaire pour l'industrie et la logistique. "
      "Utilise seulement les sources fournies. Cite les identifiants [IND-xx] ou [LOG-xx]. "
      "Si aucune source ne permet d'établir la réponse, dis-le explicitement.\n\n"
      f"CONTEXTE :\n{contexte}\n\nQUESTION : {question}\nRÉPONSE :")

q=questions_du_master[0]
meilleur = classement_rerank if reranker is not None else classement_hybride
print(prompt_documente(q['question'],meilleur(q['question'])[:3])[:1800])

## 10. Questions sans réponse et fiabilité (5 min)

Même un moteur amélioré renvoie **toujours** des voisins sur un index non vide. Cela ne signifie pas que le corpus répond à la question. Pour décider de s'abstenir, il faudrait définir et **calibrer** une politique de pertinence, éventuellement couplée à la vérification du contexte et à l'avis d'un évaluateur.

Évitez les seuils de similarité arbitraires choisis sans données de validation. Les questions suivantes ne sont pas annotées positives et ne doivent pas entrer dans le calcul de Recall@3.

In [ ]:
QUESTIONS_HORS_CORPUS = [
    'Quelle est la température réelle du moteur M-12 à cette minute ?',
    'Combien de palettes sont physiquement en stock dans le WMS aujourd’hui ?',
]
for question in QUESTIONS_HORS_CORPUS:
    print('\nQuestion sans réponse documentaire garantie :',question)
    print('Sources proposées malgré tout :',classement_hybride(question)[:3])
    print('Action attendue : signaler l’absence de données temps réel vérifiées.')

## 11. Expérimentation supplémentaire (autonomie)

Réalisez **au moins deux** expériences, puis interprétez vos observations.

1. **RRF :** faites varier `K_RRF` (10, 60, 100). Pourquoi une modification peut-elle ne rien changer sur un petit corpus ?
2. **Viviers :** faites varier `N_CANDIDATS` (4, 8, 12). Mesurez le rappel des candidats avant reranking et la latence.
3. **Parcours métier :** comparez les codes exacts (`VB-04`, `FEFO`) et les reformulations (« produit qui expire bientôt », « indice de performance des machines »).
4. **Erreurs :** trouvez un cas où le reranking dégrade MRR@3 ; expliquez-le par les passages et les annotations, sans maquiller le résultat.

**Débat scientifique :** sur une collection de 12 fiches, les mécanismes d'ANN et de passage à l'échelle ne sont pas étudiés. Pour une vraie production, il faudrait un corpus plus étendu, des filtres d'accès, un protocole de validation et des mesures de charge.

## 12. Livrables et compte rendu — par master

Rendez un **mini-rapport scientifique** comportant :

- Tableau **Recall@3, Precision@3, MRR@3, latence médiane** pour les méthodes exécutées (dont une courbe ou un graphique).
- **Deux succès** où une méthode plus avancée aide et **un échec** où elle n'aide pas (ou, s'il n'y en a pas, expliquer honnêtement cette absence).
- **Industry 4.0 :** analyse d'au moins un identifiant de maintenance et d'une paraphrase sur les données capteurs ou le TRS.
- **Logistique & services :** analyse d'au moins une question FEFO / WMS et d'une reformulation de procédure logistique.
- Une **recommandation argumentée** pour une version pilote, tenant compte des contraintes de calcul et de fiabilité.

**Grille proposée :** 30 % qualité du protocole ; 30 % métriques et résultats ; 25 % analyse critique ; 15 % reproductibilité.

Vous pouvez télécharger les deux tableaux CSV produits ci-dessous depuis le navigateur de fichiers Colab.

In [ ]:
df_eval.to_csv('TP3B_resultats_par_question.csv',index=False,encoding='utf-8-sig')
resume.to_csv('TP3B_comparaison_methodes.csv',encoding='utf-8-sig')

rapport = f"""# Compte rendu — TP 3B, optimisation du RAG

- Corpus : 12 fiches fictives, 2 masters, {len(QUESTIONS)} questions annotées
- Évaluation : k={K_EVALUATION}, candidats par méthode={N_CANDIDATS}, c_RRF={K_RRF}
- Modèle dense : {MODELE_EMBEDDING}
- Reranking : {'cross-encoder multilingue utilisé' if reranker is not None else 'non exécuté'}

## Résultats observés

{resume.to_string()}

## Discussion à compléter

1. Cas où l'hybride aide : ...
2. Cas où il dégrade le résultat : ...
3. Apport réel du reranking : ...
4. Limites : petit corpus, annotations, latence, domaine métier ...
"""
from pathlib import Path
Path('TP3B_compte_rendu_a_completer.md').write_text(rapport,encoding='utf-8')
print('Fichiers produits : TP3B_resultats_par_question.csv, TP3B_comparaison_methodes.csv, TP3B_compte_rendu_a_completer.md')

---

### Pour aller plus loin — références

- [Stéphane Robert — RAG avancé](https://www.stephane-robert.info/docs/developper/programmation/python/rag-avance/) — recherche hybride, RRF, reranking et perspective agentique.
- [Sentence Transformers — Retrieve & Re-Rank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture bi-encoder puis cross-encoder.
- [Cross-encoder multilingue mMARCO](https://huggingface.co/cross-encoder/mmarco-mMiniLMv2-L12-H384-v1) — modèle employé dans le notebook.

**Bilan pédagogique :** la meilleure méthode est celle dont les résultats sont **mesurés, interprétés et reproductibles** sur les requêtes et contraintes réellement pertinentes pour l'application visée. Aucun score du mini-corpus ne suffit pour garantir la performance d'un RAG déployé dans un site industriel ou un SI logistique.